# Stan Dirichlet-Multinomial Random Restarts

Runs Stan mean-field ADVI random restarts and plots simplex marginal moments from constrained theta draws.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REL_DIR = Path("single_MC/multinomial_dirichlet")
STAN_FILE_NAME = "stan_dirichlet_multinomial.stan"


def find_notebook_dir():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / REL_DIR]
    candidates.extend(parent for parent in cwd.parents)
    candidates.extend(parent / REL_DIR for parent in cwd.parents)
    for candidate in candidates:
        if (candidate / STAN_FILE_NAME).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not find {STAN_FILE_NAME} from {cwd}")


NOTEBOOK_DIR = find_notebook_dir()
os.chdir(NOTEBOOK_DIR)

REPO_ROOT = NOTEBOOK_DIR
while not (REPO_ROOT / "modulars").exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError("Could not locate repo root containing modulars/")
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

STAN_FILE = NOTEBOOK_DIR / STAN_FILE_NAME
print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"Stan model: {STAN_FILE}")

if Path(sys.prefix).name != "stan3":
    raise RuntimeError(
        f"This notebook must run in the miniconda environment named stan3; "
        f"current sys.prefix is {sys.prefix!r}."
    )

STAN3_PREFIX = Path(sys.prefix).resolve()
STAN3_CMDSTAN = STAN3_PREFIX / "bin" / "cmdstan"
if not STAN3_CMDSTAN.exists():
    raise FileNotFoundError(f"Expected CmdStan at {STAN3_CMDSTAN}")

os.environ["CMDSTAN"] = str(STAN3_CMDSTAN)
from cmdstanpy import cmdstan_path, set_cmdstan_path
set_cmdstan_path(str(STAN3_CMDSTAN))

print(f"Python executable: {sys.executable}")
print(f"CmdStan path: {cmdstan_path()}")


In [ ]:
from modulars.distributions import gen_multinomial_data_shared, lda_posterior_shared_theta
from modulars.utils import load_config, load_best_multid_reference

config_file = Path("dirichlet_config.json")
config = load_config(config_file)

theta_like = np.array(config["theta_like"], dtype=float)
alpha_prior = np.array(config["alpha_prior"], dtype=float)
n_cats = int(config["n_cats"])
N = int(config["N"])
total_count = int(config["total_count"])
seed = int(config.get("seed", 15))
shared = bool(config.get("shared", True))
if not shared:
    raise NotImplementedError("This Stan notebook matches the shared-theta setup used by the other methods.")

n_vec = np.full(N, total_count, dtype=int)
obs_counts = gen_multinomial_data_shared(theta_like, N, total_count, SEED=seed).astype(int)
true_post = lda_posterior_shared_theta(observations=obs_counts, alpha=alpha_prior)
best_mean, best_cov, best_std = load_best_multid_reference(
    config,
    fallback_mean=true_post["mean_post"],
    fallback_cov=np.diag(np.square(true_post["std_post"])),
)

stan_data = {
    "N": N,
    "K": n_cats,
    "total_count": n_vec,
    "y": obs_counts,
    "alpha_prior": alpha_prior,
}
PARAM_COLUMNS = [f"theta[{i}]" for i in range(1, n_cats + 1)]
LOG_COLUMNS = []
true_post


In [ ]:
from pathlib import Path

RUN_MODE = os.environ.get("SIMPLEVI_STAN_RUN_MODE", "full")  # "quick" or "full"
RUN_CONFIGS = {
    "quick": {"max_iters": 20, "n_restarts": 1, "track_every": 10, "parallel": False, "max_workers": None, "keep_outputs": True},
    "full": {"max_iters": 300_000, "n_restarts": 50, "track_every": 10, "parallel": True, "max_workers": None, "keep_outputs": False},
}

run_config = RUN_CONFIGS[RUN_MODE]
max_iters = int(os.environ.get("SIMPLEVI_STAN_MAX_ITERS", run_config["max_iters"]))
n_restarts = int(os.environ.get("SIMPLEVI_STAN_N_RESTARTS", run_config["n_restarts"]))
track_every = int(os.environ.get("SIMPLEVI_STAN_TRACK_EVERY", run_config["track_every"]))
parallel = bool(int(os.environ.get("SIMPLEVI_STAN_PARALLEL", int(run_config["parallel"]))))
max_workers_env = os.environ.get("SIMPLEVI_STAN_MAX_WORKERS")
max_workers = int(max_workers_env) if max_workers_env else run_config["max_workers"]
keep_outputs = bool(int(os.environ.get("SIMPLEVI_STAN_KEEP_OUTPUTS", int(run_config["keep_outputs"]))))

results_dir = Path(os.environ.get("SIMPLEVI_STAN_RESULTS_DIR", "results"))
results_dir.mkdir(exist_ok=True, parents=True)

run_config | {"parallel": parallel, "max_workers": max_workers, "results_dir": str(results_dir)}


In [ ]:
from modulars.stan_rr_test import run_stan_random_restarts, stan_result_tuple

stan_result = run_stan_random_restarts(
    stan_file=STAN_FILE,
    data=stan_data,
    param_columns=PARAM_COLUMNS,
    log_columns=LOG_COLUMNS,
    output_dir=results_dir / "cmdstan_runs",
    max_iters=max_iters,
    n_restarts=n_restarts,
    track_every=track_every,
    seed_offset=0,
    parallel=parallel,
    max_workers=max_workers,
    keep_outputs=keep_outputs,
    refresh=0,
)

single_means, single_stds, multi_means, multi_stds = stan_result_tuple(stan_result)
iterations = stan_result["iterations"]
print(single_means.shape, single_stds.shape, multi_means.shape, multi_stds.shape)
print(f"tracked iterations: {iterations[:5]} ... {iterations[-5:]}")
if stan_result["failures"]:
    print(f"Stan failures: {len(stan_result['failures'])}; see {results_dir / 'cmdstan_runs' / 'stan_run_failures.csv'}")


In [ ]:
from modulars import save_to_csv

save_to_csv(
    results_dir / "stan_processed_restarts.csv",
    [(single_means, single_stds, multi_means, multi_stds)],
)

results_dir


In [ ]:
from modulars import save_to_csv, load_from_csv
from modulars.plot_rr import (
    plot_simplex_dims,
    plot_dirichlet_marginals_few_restarts,
    plot_dirichlet_mean_band_rrs,
)

single_means, single_stds, multi_means, multi_stds = load_from_csv(
    results_dir / "stan_processed_restarts.csv"
)[0]

stan_summary_runs = []
for restart_idx in range(multi_means.shape[0]):
    final_mean = multi_means[restart_idx, -1, :]
    final_std = multi_stds[restart_idx, -1, :]
    stan_summary_runs.append({
        "restart_idx": restart_idx,
        "which": "multi",
        "mean": final_mean,
        "cov": np.diag(np.square(final_std)),
    })
save_to_csv(results_dir / "stan_summary_restarts.csv", stan_summary_runs)

plot_simplex_dims(
    single_means,
    single_stds,
    multi_means,
    multi_stds,
    best_mean,
    best_cov,
    param_name=r"$\theta$",
    which_dims=list(range(n_cats)),
    k=min(3, n_restarts),
    WITH_STDS=True,
    x=iterations,
)
plot_dirichlet_mean_band_rrs(
    multi_means,
    multi_stds,
    best_mean,
    best_cov=best_cov,
    param_name=r"$\theta$",
    which_dims=list(range(n_cats)),
    n_mc_samps=100,
    label_prefix="Stan ",
    x=iterations,
)
plot_dirichlet_marginals_few_restarts(
    multi_means,
    multi_stds,
    true_post["alpha_post"],
    best_mean=best_mean,
    best_cov=best_cov,
    which_restarts=list(range(min(3, n_restarts))),
    label_prefix="Stan ",
)
